In [1]:
import torch

x=torch.rand(5, 3)
print(x)

tensor([[0.4664, 0.3503, 0.0091],
        [0.0443, 0.5777, 0.4216],
        [0.1413, 0.2623, 0.2215],
        [0.3841, 0.3007, 0.0838],
        [0.3429, 0.2414, 0.2014]])


In [2]:
import torch
torch.cuda.is_available()


False

In [3]:
import torch
print(torch.device("cpu"))


cpu


In [10]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [14]:
# Download Traingin Data From open DataSets.

training_data = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [16]:
batch_size =64

#Create data Loaders.
train_dataloader = DataLoader(training_data, batch_size = batch_size)
test_dataloader = DataLoader(test_data, batch_size = batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N,C,H,W]:{X.shape}")
    print(f"Shape of y:{y.shape} {y.dtype}")
    break

Shape of X [N,C,H,W]:torch.Size([64, 1, 28, 28])
Shape of y:torch.Size([64]) torch.int64


#Creating Modles
To define a neural  network in Pytorch we create a class that inherits from 
nn.Module. We define the layers of the netwrok 
in the __init__ function and specifiy how data will pass throught he 
network in the forward function. To accelerate operations in the neural 
Network , we move it to the accelerator
such as CUDA , MPS , MTIA , or XPU. If the current accelerator is vailable , we will
use it . Otherwise, we use the CPU.
    


In [31]:
#CREATING MODELS

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define Model 
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )
    def forward(self, x):
        x= self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
model = NeuralNetwork().to(device)
print(model)
    

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [32]:
#Optimizing the Model Parameters
loss_fn= nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)  # 1e-3 ye L nahi hai 



In [33]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        X,y = X.to(device), y.to(device)
        
        #COmpute Prediction Error
        pred =  model(X)
        loss = loss_fn(pred,y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch +1) * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [29]:
def test(dataloader, model,loss_fn):
    size = len(dataloader.dataset)
    num_batches= len(dataloader)
    model.eval()
    test_loss, correct = 0, 0 
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
        test_loss /= num_batches
        correct /= size
        print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg Loss:{test_loss:>8f} \n")
    

In [34]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------------")
    train(train_dataloader,model,loss_fn,optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------------
loss: 2.307475 [   64/60000]
loss: 2.297700 [ 6464/60000]
loss: 2.279619 [12864/60000]
loss: 2.271169 [19264/60000]
loss: 2.259391 [25664/60000]
loss: 2.238307 [32064/60000]
loss: 2.243969 [38464/60000]
loss: 2.223082 [44864/60000]
loss: 2.202162 [51264/60000]
loss: 2.173304 [57664/60000]
Test Error: 
 Accuracy: 35.5%, Avg Loss:2.178045 

Epoch 2
-------------------------------------
loss: 2.185098 [   64/60000]
loss: 2.180774 [ 6464/60000]
loss: 2.128769 [12864/60000]
loss: 2.141173 [19264/60000]
loss: 2.109185 [25664/60000]
loss: 2.049570 [32064/60000]
loss: 2.078252 [38464/60000]
loss: 2.017697 [44864/60000]
loss: 2.003616 [51264/60000]
loss: 1.933326 [57664/60000]
Test Error: 
 Accuracy: 58.5%, Avg Loss:1.942293 

Epoch 3
-------------------------------------
loss: 1.972589 [   64/60000]
loss: 1.947880 [ 6464/60000]
loss: 1.839813 [12864/60000]
loss: 1.866115 [19264/60000]
loss: 1.780303 [25664/60000]
loss: 1.719850 [32064/60000]
loss: 

In [35]:
#Saving MOdels 

# A Common Way to save a model is to serialize the internal state Dictionary
# (Containing the model Parameters).

torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch model State to model.path")
           

Saved PyTorch model State to model.path


In [37]:
# Loading Models 
# The process for loading a modle includes re-creating the model structure and loading the state dictionary into it.
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [39]:
# Now model is Ready to Make predictions

classes=[
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x,y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x=x.to(device)
    pred=model(x)
    predicted,actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}",Actual: "{actual}"') 

Predicted: "Ankle boot",Actual: "Ankle boot"


Tensors
Tensors are specialized data structure that are very similar to arrays and matrices. In PyTorch , we use tensors to encode the inputs and outputs of a model , as well as the model's parameters.

Tensors are similar to NumPy's ndarrays , expect that tensors can run on GPUs or other hardware accelerators. In fact, tensor and NumPy array can oftern share the same underlying memory , eliminating the need to copy data (See Bridge with NUmpy ). Tensors are also optimized for automatic differentiation (we'll see more about that later in th Autograd section).If you are familiar with ndarrays, you 'll be right at home with the Tensor API, if not , follow along!



In [40]:
import torch
import numpy as np

In [42]:
# Initializing a Tensor
# Tensors can be initialized in various ways. Take a look at the following
# examples:
#Directly from data

#Tensors can be created directly from data. The Data Type is automatically inferred.
data=[[1,2],[3,4]]
x_data=torch.tensor(data)
    

In [43]:
#From a Numpy Array
# Tensors can be Created from NumPY arrays (and vice versa)

np_array = np.array(data)
x_np = torch.from_numpy(np_array)


In [47]:
# From Another Tensor:
# The new Tensor retains the properties (shape  ,datatype )of the argument tensor , unless explicityly overridden.

x_ones = torch.ones_like(x_data) # Retains the properties of x_data
print(f"Ones Tensor : \n{x_ones}\n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # Overrides the datatpe of  x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor : 
tensor([[1, 1],
        [1, 1]])

Random Tensor: 
 tensor([[0.1946, 0.6189],
        [0.4556, 0.3671]]) 



With Random or Constant Values 
Shape is a tuple o f tensor dimensions . In the functions below , it determines the dimensiolanlity of the output tensor.


In [48]:
shape = (2,3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor:\n {rand_tensor} \n")
print(f"Ones Tensor : \n {ones_tensor} \n")
print(f"Zeros Tenso : \n {zeros_tensor}")

Random Tensor:
 tensor([[0.2153, 0.5201, 0.2522],
        [0.5828, 0.3084, 0.2033]]) 

Ones Tensor : 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tenso : 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
